In [1]:
!pip install pytabkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 364.0/364.0 kB 6.4 MB/s eta 0:00:00


In [2]:
import gc
import random
import warnings

import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata

from pytabkit import RealMLP_TD_Classifier

warnings.filterwarnings("ignore")

SEED = 42
N_FOLDS = 3

random.seed(SEED)
np.random.seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", DEVICE)

if DEVICE != "cuda":
    print("\nWARNING: RealMLP will be much slower on CPU. Enable a Kaggle GPU if possible.")

Torch: 2.11.0+cu128
CUDA available: True
Device: cuda


In [3]:
DATA_DIR = "/kaggle/input/competitions/playground-series-s6e10"

train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
sample_submission = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")

TARGET = "satisfaction"
ID = "id"

test_ids = test[ID].copy()

if pd.api.types.is_bool_dtype(train[TARGET]):
    y = train[TARGET].astype(np.uint8).to_numpy()
elif pd.api.types.is_numeric_dtype(train[TARGET]):
    y = train[TARGET].astype(np.uint8).to_numpy()
else:
    t = train[TARGET].astype(str).str.strip().str.lower()
    vals = sorted(t.unique())

    if "satisfied" in vals:
        y = (t == "satisfied").astype(np.uint8).to_numpy()
    elif len(vals) == 2:
        mapping = {vals[0]: 0, vals[1]: 1}
        print("Target mapping:", mapping)
        y = t.map(mapping).astype(np.uint8).to_numpy()
    else:
        raise ValueError(vals)

X = train.drop(columns=[TARGET, ID]).copy()
X_test = test.drop(columns=[ID]).copy()

print("Train:", X.shape)
print("Test :", X_test.shape)
print("Positive rate:", y.mean())

Train: (699635, 21)
Test : (299844, 21)
Positive rate: 0.44357272006117476


In [4]:
# Identify original categoricals.
cat_cols = [
    c for c in X.columns
    if X[c].dtype == "object" or str(X[c].dtype).startswith("category")
]

num_cols = [c for c in X.columns if c not in cat_cols]

# Preserve categorical values as strings.
for c in cat_cols:
    X[c] = X[c].fillna("__MISSING__").astype(str)
    X_test[c] = X_test[c].fillna("__MISSING__").astype(str)

# RealMLP does not accept missing numerical values.
for c in num_cols:
    if X[c].isna().any() or X_test[c].isna().any():
        X[f"{c}__missing"] = X[c].isna().astype(np.int8)
        X_test[f"{c}__missing"] = X_test[c].isna().astype(np.int8)

        med = X[c].median()
        X[c] = X[c].fillna(med)
        X_test[c] = X_test[c].fillna(med)

print("Categoricals:", cat_cols)
print("Remaining train NaNs:", int(X.isna().sum().sum()))
print("Features:", X.shape[1])

Categoricals: ['Gender', 'Customer Type', 'Type of Travel', 'Class']
Remaining train NaNs: 0
Features: 22


In [5]:
ADD_SERVICE_CATEGORICAL_COPIES = False

SERVICE_COLS = [
    "Inflight wifi service",
    "Departure/Arrival time convenient",
    "Ease of Online booking",
    "Gate location",
    "Food and drink",
    "Online boarding",
    "Seat comfort",
    "Inflight entertainment",
    "On-board service",
    "Leg room service",
    "Baggage handling",
    "Checkin service",
    "Inflight service",
    "Cleanliness",
]

if ADD_SERVICE_CATEGORICAL_COPIES:
    for c in SERVICE_COLS:
        if c in X.columns:
            new_c = f"{c}__CAT"
            X[new_c] = X[c].astype(str)
            X_test[new_c] = X_test[c].astype(str)
            cat_cols.append(new_c)

    print("Added service categorical copies.")
else:
    print("Using plain RealMLP features for maximum model diversity.")

print("Total categorical columns:", len(cat_cols))

Using plain RealMLP features for maximum model diversity.
Total categorical columns: 4


In [6]:
skf = StratifiedKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=SEED
)

oof_real = np.zeros(len(X), dtype=np.float32)
pred_real = np.zeros(len(X_test), dtype=np.float32)
fold_scores = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), start=1):
    print(f"\n========== RealMLP fold {fold}/{N_FOLDS} ==========")

    model = RealMLP_TD_Classifier(
        device=DEVICE,
        random_state=SEED + fold,
        n_cv=1,
        n_refit=0,

        # Runtime-conscious first pass.
        n_epochs=128,
        batch_size=1024,

        # AUC-oriented validation.
        val_metric_name="1-auc_ovr",
        use_ls=False,

        verbosity=1
    )

    X_tr = X.iloc[tr_idx].reset_index(drop=True)
    X_va = X.iloc[va_idx].reset_index(drop=True)
    y_tr = y[tr_idx]
    y_va = y[va_idx]

    model.fit(
        X_tr,
        y_tr,
        X_va,
        y_va,
        cat_col_names=cat_cols
    )

    va_pred = model.predict_proba(X_va)[:, 1]
    te_pred = model.predict_proba(X_test)[:, 1]

    oof_real[va_idx] = va_pred.astype(np.float32)
    pred_real += te_pred.astype(np.float32) / N_FOLDS

    auc = roc_auc_score(y_va, va_pred)
    fold_scores.append(auc)

    print(f"Fold {fold} AUC: {auc:.6f}")

    del model, X_tr, X_va, y_tr, y_va, va_pred, te_pred
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

real_auc = roc_auc_score(y, oof_real)

print("\n========================================")
print("Fold AUCs:", [round(x, 6) for x in fold_scores])
print("RealMLP OOF AUC:", round(float(real_auc), 6))
print("========================================")


========== RealMLP fold 1/3 ==========
Columns classified as continuous: ['Age', 'Flight Distance', 'Inflight wifi service', 'Departure/Arrival time convenient', 'Ease of Online booking', 'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort', 'Inflight entertainment', 'On-board service', 'Leg room service', 'Baggage handling', 'Checkin service', 'Cleanliness', 'Departure Delay in Minutes', 'Arrival Delay in Minutes', 'Arrival Delay in Minutes__missing']
Columns classified as categorical: ['Gender', 'Customer Type', 'Type of Travel', 'Class']


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
`Trainer.fit` stopped: `max_epochs=128` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics 

Fold 1 AUC: 0.957570

========== RealMLP fold 2/3 ==========
Columns classified as continuous: ['Age', 'Flight Distance', 'Inflight wifi service', 'Departure/Arrival time convenient', 'Ease of Online booking', 'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort', 'Inflight entertainment', 'On-board service', 'Leg room service', 'Baggage handling', 'Checkin service', 'Cleanliness', 'Departure Delay in Minutes', 'Arrival Delay in Minutes', 'Arrival Delay in Minutes__missing']
Columns classified as categorical: ['Gender', 'Customer Type', 'Type of Travel', 'Class']


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
`Trainer.fit` stopped: `max_epochs=128` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics 

Fold 2 AUC: 0.958544

========== RealMLP fold 3/3 ==========
Columns classified as continuous: ['Age', 'Flight Distance', 'Inflight wifi service', 'Departure/Arrival time convenient', 'Ease of Online booking', 'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort', 'Inflight entertainment', 'On-board service', 'Leg room service', 'Baggage handling', 'Checkin service', 'Cleanliness', 'Departure Delay in Minutes', 'Arrival Delay in Minutes', 'Arrival Delay in Minutes__missing']
Columns classified as categorical: ['Gender', 'Customer Type', 'Type of Travel', 'Class']


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
`Trainer.fit` stopped: `max_epochs=128` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics 

Fold 3 AUC: 0.958258

Fold AUCs: [np.float64(0.95757), np.float64(0.958544), np.float64(0.958258)]
RealMLP OOF AUC: 0.958106


In [7]:
submission = sample_submission.copy()
submission[ID] = test_ids.values
submission[TARGET] = np.clip(pred_real, 1e-7, 1 - 1e-7)

submission.to_csv("submission_realmlp.csv", index=False)

np.save("oof_realmlp.npy", oof_real)
np.save("pred_realmlp.npy", pred_real)

print(submission.head())
print("\nSaved:")
print("  submission_realmlp.csv")
print("  oof_realmlp.npy")
print("  pred_realmlp.npy")

       id  satisfaction
0  699635      0.258929
1  699636      0.827278
2  699637      0.929757
3  699638      0.069016
4  699639      0.047948

Saved:
  submission_realmlp.csv
  oof_realmlp.npy
  pred_realmlp.npy


In [8]:
from pathlib import Path

needed = [
    "oof_lgb_v2.npy",
    "pred_lgb_v2.npy",
    "oof_cat_v2.npy",
    "pred_cat_v2.npy",
]

if all(Path(f).exists() for f in needed):
    oof_lgb = np.load("oof_lgb_v2.npy")
    pred_lgb = np.load("pred_lgb_v2.npy")
    oof_cat = np.load("oof_cat_v2.npy")
    pred_cat = np.load("pred_cat_v2.npy")

    print("LGB AUC :", roc_auc_score(y, oof_lgb))
    print("CAT AUC :", roc_auc_score(y, oof_cat))
    print("REAL AUC:", roc_auc_score(y, oof_real))

    # Convert all model outputs to percentile ranks.
    r_lgb = rankdata(oof_lgb) / len(oof_lgb)
    r_cat = rankdata(oof_cat) / len(oof_cat)
    r_real = rankdata(oof_real) / len(oof_real)

    rt_lgb = rankdata(pred_lgb) / len(pred_lgb)
    rt_cat = rankdata(pred_cat) / len(pred_cat)
    rt_real = rankdata(pred_real) / len(pred_real)

    best_auc = -1
    best_w = None

    # 0.05 grid; weights sum to 1.
    grid = np.arange(0, 1.0001, 0.05)

    for w_lgb in grid:
        for w_cat in grid:
            w_real = 1.0 - w_lgb - w_cat

            if w_real < -1e-9:
                continue

            blend = (
                w_lgb * r_lgb +
                w_cat * r_cat +
                w_real * r_real
            )

            auc = roc_auc_score(y, blend)

            if auc > best_auc:
                best_auc = auc
                best_w = (float(w_lgb), float(w_cat), float(w_real))

    print("\nBest 3-model rank blend")
    print("OOF AUC:", round(float(best_auc), 6))
    print(
        "Weights:",
        {"LGB": best_w[0], "CAT": best_w[1], "RealMLP": best_w[2]}
    )

    final_pred = (
        best_w[0] * rt_lgb +
        best_w[1] * rt_cat +
        best_w[2] * rt_real
    )

    blend_submission = sample_submission.copy()
    blend_submission[ID] = test_ids.values
    blend_submission[TARGET] = final_pred
    blend_submission.to_csv("submission_v3_realmlp_blend.csv", index=False)

    print("Saved: submission_v3_realmlp_blend.csv")

else:
    print("V2 .npy files not found in this session.")
    print("Run/upload those prediction arrays if you want automatic 3-model blending.")

V2 .npy files not found in this session.
Run/upload those prediction arrays if you want automatic 3-model blending.
